# RMSO-BW 迟滞建模与 Fuzzy-NN 补偿控制演示

本 Notebook 演示了论文 *"Intelligent Rate-Dependent Hysteresis Control Compensator Design With Bouc-Wen Model Based on RMSO for Piezoelectric Actuator"* (Liu et al., 2020) 中提出的控制框架的完整流程。

包含以下步骤：
1. **数据生成**：生成带有频率相关迟滞特性的合成压电执行器（PEA）数据。
2. **参数辨识**：使用基于区域的混合种群粒子群优化算法 (RMSO) 辨识 Bouc-Wen 迟滞模型参数。
3. **模型评估**：对比辨识模型输出与真实迟滞环。
4. **自适应控制**：结合辨识的 BW 前馈逆模型和在线 Fuzzy-NN 反馈控制器，进行轨迹跟踪补偿仿真。

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from rmso_bw.src.rmso_bw_model import RMSO_BW_Model
from rmso_bw.src.fuzzy_nn_controller import FuzzyNNController
from rmso_bw.src.rmso_bw_compensator import RMSO_BW_Compensator

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)

## 1. 生成合成迟滞数据

In [ ]:
# 设定"真实"的 Bouc-Wen 参数作为被控对象 (Plant)
true_params = {
    'A': 2.5,
    'alpha': 0.8,
    'beta': 0.05,
    'gamma': 0.05,
    'n': 1.0
}
plant_model = RMSO_BW_Model(params=true_params)

# 提取率相关的迟滞环用于辨识 (使用 5Hz 正弦波)
freq = 5.0
dt = 0.001
t_ident = np.arange(0, 2.0/freq, dt) # 2个周期
u_ident = 5.0 * np.sin(2 * np.pi * freq * t_ident)
y_ident = plant_model.simulate(u_ident, dt)

# 为了模拟真实情况，在辨识数据中加入少量噪声
y_meas = y_ident + np.random.normal(0, 0.05, size=y_ident.shape)

plt.figure(figsize=(8, 5))
plt.plot(u_ident, y_meas, 'k.', markersize=2, label='Measured Data')
plt.plot(u_ident, y_ident, 'r-', alpha=0.7, label='True Hysteresis')
plt.title('Synthetic Hysteresis Data for Identification')
plt.xlabel('Input Voltage (V)')
plt.ylabel('Output Displacement (um)')
plt.legend()
plt.grid(True)
plt.show()

## 2. 使用 RMSO 算法进行参数辨识

In [ ]:
# 初始化待辨识模型
est_model = RMSO_BW_Model()

# 设定搜索边界
bounds = {
    'A': (0.1, 5.0),
    'alpha': (0.1, 2.0),
    'beta': (0.01, 0.2),
    'gamma': (0.01, 0.2),
    'n': (1.0, 2.0)  # n通常固定为1或2附近
}

# 运行 RMSO 优化 (此处减少迭代次数以加快演示速度，论文中可能使用 200+ 次)
print("Starting RMSO Identification...")
identified_params = est_model.identify_with_rmso(
    u_meas=u_ident, 
    y_meas=y_meas, 
    dt=dt, 
    bounds=bounds, 
    pop_size=50, 
    max_iter=50
)
print("\nTrue Params:", true_params)
print("Identified:", {k: round(v, 4) for k, v in identified_params.items()})

## 3. 模型验证与可视化

In [ ]:
# 使用辨识得到的模型重新模拟输出
y_sim = est_model.simulate(u_ident, dt)

plt.figure(figsize=(8, 5))
plt.plot(u_ident, y_meas, 'k.', markersize=2, label='Measured Data')
plt.plot(u_ident, y_sim, 'b-', linewidth=2, label='Identified BW Model')
plt.title('Bouc-Wen Model Parameter Identification Result')
plt.xlabel('Input Voltage (V)')
plt.ylabel('Output Displacement (um)')
plt.legend()
plt.grid(True)
plt.show()

## 4. RMSO-BW + Fuzzy-NN 闭环补偿控制

In [ ]:
# 定义复杂的期望跟踪轨迹 (多频混合信号)
t_track = np.arange(0, 1.0, dt)
desired_traj = 3.0 * np.sin(2 * np.pi * 2 * t_track) + 1.5 * np.sin(2 * np.pi * 10 * t_track)

# 定义真实的被控对象函数
def true_plant(u, dt):
    # 简单使用全局的 plant_model 来获取每一步的输出。
    # 由于 plant_model.simulate 接受序列，我们为了符合控制接口需要写一个在线式的闭环调用。
    # 为了简化 Notebook 演示，直接将其作为黑盒连续传入序列模拟。
    pass

# 为适应连续闭环，我们将 plant 改写为一个维护状态的闭包
def create_online_plant(params):
    state = {'h': 0.0}
    A, beta, gamma, n, alpha = params['A'], params['beta'], params['gamma'], params['n'], params['alpha']
    last_u = 0.0
    
    def plant_step(u, dt):
        u_dot = (u - last_u) / dt
        term1 = A * u_dot
        term2 = beta * abs(u_dot) * state['h'] * (abs(state['h']) ** (n - 1))
        term3 = gamma * u_dot * (abs(state['h']) ** n)
        
        h_dot = term1 - term2 - term3
        state['h'] += h_dot * dt
        
        # 模拟外部扰动和未建模动态
        disturbance = 0.1 * np.sin(2*np.pi*50*t_track[0]) # 暂定用常数扰动占位
        y = alpha * u + state['h'] + np.random.normal(0, 0.02)
        return y
    
    return plant_step

plant = create_online_plant(true_params)

# 初始化模糊神经网络控制器 (单输入: error)
fuzzy_ctrl = FuzzyNNController(n_rules=7, n_inputs=1)

# 初始化整体补偿器 (使用辨识到的模型和模糊控制器)
compensator = RMSO_BW_Compensator(bw_model=est_model, fuzzy_controller=fuzzy_ctrl)

# 运行闭环跟踪仿真
time_seq, desired_traj, actual_traj = compensator.track(
    desired_trajectory=desired_traj, 
    plant_fn=plant, 
    dt=dt, 
    online_learning_rate=0.05
)

compensator.plot_tracking(time_seq, desired_traj, actual_traj)